# 강의 04 · 실습 3 — 서브그래프 모듈화 · (5) 고난도 II

## 1. 문제상황

- 온라인 서점 고객센터의 메일 처리 그래프에 요구가 두 가지 더 붙었습니다.
- 첫째, 분류 절차는 분류 팀이, 긴급도 판정 절차는 운영 팀이 따로 관리하고 싶어 합니다. 두 팀은 서로의 코드를 건드리지 않기를 원합니다.
- 둘째, 두 팀은 고객센터 그래프와 같은 상태 키(email·category·priority·handled)를 쓰기로 이미 합의했습니다. 키를 번역할 필요가 없습니다.
- 지금 그래프는 분류와 긴급도 판정이 한 노드 함수 안에 함께 적혀 있어서, 한 팀이 고치면 다른 팀의 코드까지 다시 확인해야 합니다.
- 긴급도 판정은 분류 결과를 읽어야 하므로, 분류가 끝난 뒤에만 실행되어야 합니다.

## 2. 문제와 목표

- **문제**: 두 팀이 따로 관리해야 할 절차가 한 노드 안에 섞여 있고, 순서(분류 뒤 긴급도)를 지켜야 합니다.
- **목표**
  - 분류 절차와 긴급도 판정 절차를 각각 따로 컴파일한 자식 그래프로 만듭니다.
    - 자식 둘: 첫 자식은 메일 본문의 앞뒤 공백을 지우고 줄바꿈을 띄어쓰기로 바꾼 뒤 분류하고, 둘째 자식은 긴급도를 판정합니다.
    - 긴급 판정의 기준: 분류 결과가 파손이거나 본문에 「급」이 들어 있으면 긴급, 아니면 일반
  - 고객센터 부모 그래프가 두 자식을 순서대로 거친 뒤 담당자를 배정하게 만듭니다.
  - 두 팀과 고객센터가 같은 상태 키를 쓰므로 키를 번역하는 코드는 두지 않습니다.
  - 메일 두 통의 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**:
  - 찢어진 책 메일과 배송 조회 메일을 넣었을 때, 첫 메일은 「파손 담당자에게 긴급 배정」, 둘째 메일은 「배송 담당자에게 일반 배정」이 되고,
  - 부모의 실행 결과에는 자식 안의 노드 이름이 나타나지 않고, 두 자식은 각각 단독으로도 실행되는 것을 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex03_s5_diagram.svg)

## 4. 단계별 요구사항

(다이어그램을 보고 요구사항을 번호 목록으로 적습니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리를 불러오고 모델을 준비합니다. 아래 코드 셀은 채워져 있으므로 그대로 실행합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")


# 주어진 자료
EMAILS = [   # 1번 메일의 공백·줄바꿈은 일부러 넣은 것 — 그대로 둔다
    "주문한 책이   찢어진 채로\n왔습니다. 내일 선물해야 해서 급합니다.",
    "배송 조회가 되지 않습니다. 언제쯤 도착하나요?",
]


In [ ]:
# 여기에 「5. 코드 골격」의 표 순서대로 코드를 작성합니다. 단계마다 셀을 나눕니다.

## 7. 실행 결과 확인

스스로 작성한 코드의 실행 결과에서 다음 세 가지를 확인합니다.

1. 두 자식 그래프를 각각 단독으로 실행할 수 있습니다. 첫 자식은 분류 결과를, 둘째 자식은 손으로 넣은 분류 결과를 읽어 긴급도를 돌려줍니다.
2. 부모의 실행 결과에는 부모에 등록한 노드 이름만 출력되고, 자식 안의 노드 이름은 나타나지 않습니다.
3. 1번 메일의 최종 상태는 「파손 담당자에게 긴급 배정」, 2번 메일은 「배송 담당자에게 일반 배정」입니다. 둘째 자식이 읽은 분류 결과는 첫 자식이 공용 상태에 써 둔 값입니다.